Copyright Matlantis Corp. as contributors to Matlantis contrib project

# Steered Molecular Dynamics (SMD) Simulations

このノートブックは、Deca-alanineのヘリックス構造に対して **Steered Molecular Dynamics (SMD)** を実行することで、アンブレラサンプリングの各ウィンドウの初期構造を準備します。

SMDは、分子や原子に外力を加えて強制的に構造変化を引き起こす手法です。ここでは以下の手順で計算を行います。

1.  **集団変数 (Collective Variable: CV) の定義:**
    分子の両端の原子間距離を集団変数 (反応座標)として定義します。
2.  **移動拘束 (Moving Restraint):**
    定義した集団変数について、調和振動子(バネ)による拘束をかけます。拘束位置を時間とともに徐々に変化させることで、分子や原子を徐々に引っ張り、強制的に構造変化を引き起こします。

**注意:**
このNotebookは **PLUMED** がインストールされた環境でのみ動作します。


## Step 1. PLUMED環境の設定

Steered MDを行う際に使用するPLUMEDは外部ライブラリであるため、Pythonからカーネルを正しく呼び出せるようにパスと環境変数を設定します。
※ 環境に合わせて `PLUMED_ROOT` のパスを適宜修正してください。

In [ ]:
# PLUMED environment
import os
import sys

# PLUMEDへのパス設定
PLUMED_ROOT   = os.path.expanduser("~/local/plumed-2.9.0")  # 必要に応じてplumedをインストールしたdirに修正する
plumed_bin    = os.path.join(PLUMED_ROOT, "bin")
plumed_lib    = os.path.join(PLUMED_ROOT, "lib")
plumed_kernel = os.path.join(plumed_lib, "libplumedKernel.so")

# 環境変数の設定
os.environ["PATH"]            = f"{plumed_bin}:{os.environ.get('PATH', '')}"
os.environ["LD_LIBRARY_PATH"] = f"{plumed_lib}:{os.environ.get('LD_LIBRARY_PATH', '')}"
os.environ["PLUMED_KERNEL"]   = str(plumed_kernel)

# Pythonライブラリパスの追加
if str(PLUMED_ROOT) not in sys.path:
    sys.path.append(str(PLUMED_ROOT))

## Step2: ライブラリのインポート、PFPの設定

In [ ]:
import numpy as np

# ASE
from ase import units
from ase.io import write
from ase.io.proteindatabank import read_proteindatabank
from ase.md.langevin import Langevin
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution, Stationary
from ase.calculators.plumed import Plumed

# PFP
from pfp_api_client.pfp.calculators.ase_calculator import ASECalculator
from pfp_api_client.pfp.estimator import Estimator

calc_mode = "R2SCAN_PLUS_D3"
method_type = "PFVM_D3_PFVM"
model_version = "v8.0.0"
estimator = Estimator(calc_mode=calc_mode, method_type=method_type, model_version=model_version)
calculator = ASECalculator(estimator)

## Step 3. 初期構造の読み込み

前のノートブック [(NVTアンサンブルの分子動力学シミュレーション)](./02_equilibrium_nvt_md_ja.ipynb)で緩和させた構造 `deca_alanine_helix_eq.pdb` を読み込みます。平衡化過程をスキップしている場合やファイルが見つからない場合は、`assets`に配置している参照用構造を使用します。

また、本Notebookの結果を出力するディレクトリ (`./output/03_steered_md`) をここで作成しておきます。

In [ ]:
# インプットファイル
if os.path.exists(f"./output/02_equilibrium_nvt_md/deca_alanine_helix_eq.pdb"): # 02平衡化MDを実施している場合
    inp_pdb = "./output/02_equilibrium_nvt_md/deca_alanine_helix_eq.pdb"
else:
    inp_pdb = f"./assets/03_steered_md/deca_alanine_helix_eq.pdb"

# アウトプットディレクトリ
out_root = './output/03_steered_md'
os.makedirs(out_root, exist_ok=True)

In [ ]:
# PDBを読み込む
atoms = read_proteindatabank(inp_pdb)
atoms

## Step 4. SMDのスケジュール設定 (ターゲット距離の作成)

読み込んだ初期構造（平衡状態）の末端原子間距離を基準として、圧縮と伸長の2方向へターゲット距離を変化させていきます。

* **圧縮方向 (`cv_compression`)**: ヘリックスがつぶれた状態へと、末端間距離を短くしていく。
* **伸長方向 (`cv_stretch`)**: ヘリックスが伸び切った鎖状の状態へと末端間距離を伸ばしてく。

ここでは 0.05 Å 刻み (`d_step`) で末端間距離を変化させます。急激に距離を変化させると系が破綻したり、予期せない構造変化経路をたどるリスクがあるため、小さなステップで徐々に引き延ばす（あるいは圧縮する）ことがポイントとなります。

In [ ]:
# Steered MDの制御パラメータ
d_step  = 0.05   # 距離の刻み幅 [Å]

# 原子インデックス (両端のCA原子間の距離を制御)
atom_idx_1 = 8   # N末端側のCA原子
atom_idx_2 = 98  # C末端側のCA原子

# 現在の距離を取得
d_start = round(atoms.get_distance(atom_idx_1, atom_idx_2) / d_step) * d_step

# 探索範囲の定義
d_min   = 12.99
d_max   = 38.01

# 各ウインドウのtime steps
steps_window = 1_000

# ターゲット距離リスト作成
cv_compression = np.arange(d_start, d_min, -d_step)
cv_stretch     = np.arange(d_start, d_max,  d_step)[1:]

print(f"Start Distance: {d_start:.2f} A")
print(f"Range: {d_min} A <-> {d_max} A")

## Step 5. Steered MDの実行
### 5-1. 圧縮方向のSteered MD実行

定義したスケジュール (`cv_compression`) に従い、分子を圧縮する方向に連続的にシミュレーションを行います。

* **PLUMEDの設定**:
    * 2原子間の距離 (`DISTANCE`) を定義します。
    * その距離に対し、調和ポテンシャル (`RESTRAINT`) をかけます。バネ定数 `KAPPA` と中心値 `AT` を指定します。
    * バネ定数や中心値の単位系は`UNITS`でeVとÅに指定しています。

* **シミュレーションの流れ**:
    1.  各ターゲット距離 (`AT`) で 1,000 ステップの Langevin ダイナミクスを実行。
    2.  最終構造を `md-dyn-restart.pdb` として保存。
    3.  次のループでは前のウインドウの最終構造を初期構造とします (つまり、同一の`atoms` オブジェクトが更新され続けます）。

In [ ]:
for cv in cv_compression:

    # ------------------------------------------------------------
    # Prepare Directory
    # ------------------------------------------------------------
    cv_str = f"{cv:.2f}"
    out_dir = f"{out_root}/cv_{cv_str}"
    os.makedirs(out_dir, exist_ok=True)

    # ------------------------------------------------------------
    # PLUMED Settingsof Collective Variables
    # ------------------------------------------------------------

    # PLUMEDは 1-based index なので、Pythonのindexに+1する
    plumed_idx_1 = atom_idx_1 + 1
    plumed_idx_2 = atom_idx_2 + 1

    plumed_setting = [
        f"UNITS LENGTH=A ENERGY=eV",

        # 1. Define distance
        f"dist: DISTANCE ATOMS={plumed_idx_1},{plumed_idx_2}",

        # 2. Apply umbrella potential (https://www.plumed.org/doc-v2.9/user-doc/html/lugano-2.html)
        f"restraint: RESTRAINT ARG=dist KAPPA=0.2 AT={cv_str}",

        # 3. Output
        f"PRINT STRIDE=100 ARG=dist,restraint.bias,restraint.force2 FILE={out_dir}/COLVAR_{cv_str}",
        "FLUSH STRIDE=1000"
    ]

    print(plumed_setting)

    # ------------------------------------------------------------
    # Molecular Dynamics
    # ------------------------------------------------------------
    timestep = 1.0 * units.fs
    temperature = 300.0

    # PLUMED calculator
    atoms.calc = Plumed(calc=calculator, input=plumed_setting, timestep=timestep, atoms=atoms, kT=1)

    # Set the momenta corresponding to the given "temperature"
    MaxwellBoltzmannDistribution(atoms, temperature_K=temperature,force_temp=True)
    Stationary(atoms)  # Set zero total momentum to avoid drifting

    # Dynamics
    dyn = Langevin(atoms,
                   timestep,
                   temperature_K=temperature,
                   friction=0.002/units.fs,
                   trajectory=f'{out_dir}/md-dyn.traj',
                   logfile=f'{out_dir}/md-dyn.log',
                   loginterval=100)

    dyn.run(steps_window)

    write(f'{out_dir}/md-dyn-restart.pdb', atoms)

### 5-2. 伸長方向のSteered MD実行

定義したスケジュール (`cv_stretch`) に従い、分子を伸長する方向に引っ張るシミュレーションを行います。

In [ ]:
# 初期構造をリロードする
atoms = read_proteindatabank(inp_pdb)

for cv in cv_stretch:

    # ------------------------------------------------------------
    # Prepare Directory
    # ------------------------------------------------------------
    cv_str = f"{cv:.2f}"
    out_dir = f"{out_root}/cv_{cv_str}"
    os.makedirs(out_dir, exist_ok=True)

    # ------------------------------------------------------------
    # PLUMED Settings　of Collective Variables
    # ------------------------------------------------------------

    # PLUMEDは 1-based index なので、Pythonのindexに+1する
    plumed_idx_1 = atom_idx_1 + 1
    plumed_idx_2 = atom_idx_2 + 1

    plumed_setting = [
        f"UNITS LENGTH=A ENERGY=eV",

        # 1. Define distance
        f"dist: DISTANCE ATOMS={plumed_idx_1},{plumed_idx_2}",

        # 2. Apply umbrella potential (https://www.plumed.org/doc-v2.9/user-doc/html/lugano-2.html)
        f"restraint: RESTRAINT ARG=dist KAPPA=0.2 AT={cv_str}",

        # 3. Output
        f"PRINT STRIDE=100 ARG=dist,restraint.bias,restraint.force2 FILE={out_dir}/COLVAR_{cv_str}",
        "FLUSH STRIDE=1000"
    ]

    print(plumed_setting)

    # ------------------------------------------------------------
    # Molecular Dynamics
    # ------------------------------------------------------------
    timestep = 1.0 * units.fs
    temperature = 300.0

    # PLUMED calculator
    atoms.calc = Plumed(calc=calculator, input=plumed_setting, timestep=timestep, atoms=atoms, kT=1)

    # Set the momenta corresponding to the given "temperature"
    MaxwellBoltzmannDistribution(atoms, temperature_K=temperature,force_temp=True)
    Stationary(atoms)  # Set zero total momentum to avoid drifting

    # Dynamics
    dyn = Langevin(atoms,
                   timestep,
                   temperature_K=temperature,
                   friction=0.002/units.fs,
                   trajectory=f'{out_dir}/md-dyn.traj',
                   logfile=f'{out_dir}/md-dyn.log',
                   loginterval=100)

    dyn.run(steps_window)

    write(f'{out_dir}/md-dyn-restart.pdb', atoms)

### Next Step

Steered MDによって、圧縮状態から伸長状態まで、連続的に構造を変化させたデータセットが得られました。

次の[04_select_umbrella_sampling_initial_structures](./04_select_umbrella_sampling_initial_structures_ja.ipynb) では、得られたトラジェクトリー構造の中から、アンブレラサンプリング計算に適した初期構造を抜き出す作業を行います。